In [1]:
print("ok")

ok


In [2]:
%pwd
import os
os.chdir("../")

In [3]:
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


c:\Users\Anshul Parkar\Desktop\MindBridge\ai-backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# %pip install pandas

In [5]:
# Load the CSV file
csv_path = "csvData/quesAns.csv"


In [6]:
import pandas as pd

pd.read_csv(csv_path, nrows=1)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
loader = CSVLoader(
    file_path=csv_path,
    source_column="Context",      # column name for question
    content_columns="Response"  # column name for answer
)

In [ ]:
documents = loader.load()

In [ ]:
print(f"Loaded {len(documents)} question-answer pairs")

In [ ]:
def text_split(docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    return splitter.split_documents(docs)

text_chunks = text_split(documents)
print("Number of chunks:", len(text_chunks))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

import os
from dotenv import load_dotenv
load_dotenv()
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "healthai-csv"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
index = pc.Index(index_name)

In [ ]:
print(index)

In [ ]:
from langchain_pinecone import Pinecone as PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)


In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [ ]:
from langchain_groq import ChatGroq
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.4,
    max_tokens=500,
    api_key=GROQ_API_KEY
)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [ ]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise.\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke("How can I change my feeling of being worthless to everyone?")
print("Answer:", response)